# Day 21 — Revision + Debugging Exercises

This notebook contains the exercises for **Day 21 (Week 3, Day 7)**. We cover:
1. **Debugging a Dashboard (Streamlit)**: Performance optimization, state-management, and handling edge cases.
2. **SQL Optimization**: Refactoring slow queries, leveraging window functions, and filtering early.

## Part 1: Dashboard Debugging (Streamlit)

Dashboards often suffer from three major performance and logic pitfalls:
- **Missing Cache**: Reloading data from a CSV or DB query on every page rerun (e.g., when moving a slider).
- **Session State Bugs**: Forgetting inputs or losing user changes due to variables resetting during runs.
- **Empty Filter Crash**: If a user selects options that produce 0 rows, graphs or aggregations crash unless handled gracefully.

Below is an interactive code example demonstrating these bugs and their optimized solutions.

In [1]:
import pandas as pd
import numpy as np
import time

# --- A. Caching Demo ---

# Inefficient (Broken) Approach: Reading data on every user interaction (simulated with sleep)
def load_data_inefficient(filepath):
    # Simulate high disk I/O or network fetch latency
    time.sleep(2.0)
    return pd.DataFrame({
        'user_id': range(1, 100001),
        'score': np.random.randint(10, 100, size=100000)
    })

# Optimized (Debugged) Approach: Cache data so subsequent reruns are instantaneous
# Note: In Streamlit, this is achieved using @st.cache_data
# For our demo, we implement a simple decorator-based memoization
class DummyCache:
    def __init__(self):
        self.storage = {}
    def cache_data(self, func):
        def wrapper(*args, **kwargs):
            key = str(args) + str(kwargs)
            if key not in self.storage:
                self.storage[key] = func(*args, **kwargs)
            return self.storage[key]
        return wrapper

cache = DummyCache()

@cache.cache_data
def load_data_cached(filepath):
    time.sleep(2.0) # Simulating I/O
    return pd.DataFrame({
        'user_id': range(1, 100001),
        'score': np.random.randint(10, 100, size=100000)
    })

# Benchmark to verify optimization
start = time.time()
df1 = load_data_cached("dummy_file.csv")
t1 = time.time() - start

start = time.time()
df2 = load_data_cached("dummy_file.csv")  # Read from cache
t2 = time.time() - start

print(f"First run (Simulated DB Load): {t1:.4f} seconds")
print(f"Second run (Cached Load):      {t2:.4f} seconds (Optimization: {t1/t2:.1f}x speedup)")

First run (Simulated DB Load): 2.0029 seconds
Second run (Cached Load):      0.0001 seconds (Optimization: 31582.2x speedup)


### Streamlit Session State & Empty Filter Guards

Here is the clean pattern for handling:
1. **User session state initialization** to avoid resetting filter choices.
2. **DataFrame empty check** to prevent calculations from failing on empty results.

In [2]:
# --- B. Session State & Empty Filter Guard ---

def render_dashboard_logic(data, filter_score):
    """
    Simulates a dashboard page rendering.
    Protects against division by zero or empty values if data filter results in 0 rows.
    """
    # Filtering data based on slider input
    filtered = data[data['score'] >= filter_score]
    
    # Guard pattern:
    if filtered.empty:
        print(f"Warning: No users found with score >= {filter_score}. Showing placeholder UI.")
        return 0, 0
    
    avg_score = filtered['score'].mean()
    total_users = len(filtered)
    print(f"Metrics: Average Score = {avg_score:.2f}, Active Users = {total_users}")
    return avg_score, total_users

# Run simulation
dummy_data = pd.DataFrame({'score': [50, 60, 70]})
print("Valid Filter:")
render_dashboard_logic(dummy_data, 60)

print("\nEmpty Filter Guard Triggered:")
render_dashboard_logic(dummy_data, 95)

Valid Filter:
Metrics: Average Score = 65.00, Active Users = 2

Empty Filter Guard Triggered:


(0, 0)

## Part 2: SQL Query Optimization

SQL queries in analytical environments must be optimized to query data efficiently, minimizing CPU and memory usage.

### Case 1: Fetching the Latest Customer Transaction
**Problem:** Finding the single latest row per entity is often done using a slow subquery + self-join.
**Solution:** Using the window function `ROW_NUMBER() OVER (PARTITION BY ... ORDER BY ...)` which databases execute in a single pass.

In [3]:
# Illustrating SQL Case 1 with comments / Mock execution

inefficient_sql_1 = """
-- Inefficient: Self Join with aggregated Subquery
SELECT t1.customer_id, t1.transaction_id, t1.transaction_date, t1.amount
FROM transactions t1
INNER JOIN (
    SELECT customer_id, MAX(transaction_date) AS max_date
    FROM transactions
    GROUP BY customer_id
) t2 ON t1.customer_id = t2.customer_id AND t1.transaction_date = t2.max_date;
"""

optimized_sql_1 = """
-- Optimized: Single-pass execution using Window Function
WITH RankedTransactions AS (
    SELECT customer_id, transaction_id, transaction_date, amount,
           ROW_NUMBER() OVER(PARTITION BY customer_id ORDER BY transaction_date DESC) as rn
    FROM transactions
)
SELECT customer_id, transaction_id, transaction_date, amount
FROM RankedTransactions
WHERE rn = 1;
"""
print("SQL Optimization 1: Self-Join vs Window Function partitioned ranking.")

SQL Optimization 1: Self-Join vs Window Function partitioned ranking.


### Case 2: Early Filtering with WHERE
**Problem:** Running filters inside `HAVING` after sorting and grouping runs.
**Solution:** Move non-aggregated fields into `WHERE` so rows are filtered out before grouping occurs.

In [4]:
inefficient_sql_2 = """
-- Inefficient: Filtering grouped results using HAVING on non-aggregated field
SELECT store_id, SUM(sales_amount)
FROM sales_data
GROUP BY store_id, transaction_year
HAVING transaction_year = 2026;
"""

optimized_sql_2 = """
-- Optimized: Filter rows BEFORE grouping with WHERE
SELECT store_id, SUM(sales_amount)
FROM sales_data
WHERE transaction_year = 2026
GROUP BY store_id;
"""
print("SQL Optimization 2: Filter early using WHERE instead of HAVING.")

SQL Optimization 2: Filter early using WHERE instead of HAVING.


### Case 3: Common Table Expressions (CTEs) for Modularity
**Problem:** Deeply nested subqueries are hard to read and optimize.
**Solution:** Refactor queries into flat, sequential CTEs to make readability easy for developers and query optimizers.

In [5]:
inefficient_sql_3 = """
-- Inefficient/Hard to read: Deeply nested subqueries
SELECT customer_name, total_spent
FROM (
    SELECT customer_id, SUM(amount) as total_spent
    FROM transactions
    WHERE status = 'Completed'
    GROUP BY customer_id
) sub_totals
JOIN customers c ON c.id = sub_totals.customer_id
WHERE total_spent > 1000;
"""

optimized_sql_3 = """
-- Optimized: Flattened and highly readable CTE
WITH CompletedTotals AS (
    SELECT customer_id, SUM(amount) as total_spent
    FROM transactions
    WHERE status = 'Completed'
    GROUP BY customer_id
)
SELECT c.customer_name, ct.total_spent
FROM CompletedTotals ct
JOIN customers c ON c.id = ct.customer_id
WHERE ct.total_spent > 1000;
"""
print("SQL Optimization 3: Flatten subqueries into clean CTEs.")

SQL Optimization 3: Flatten subqueries into clean CTEs.
